<a href="https://colab.research.google.com/github/Rifana63/ECG-1/blob/main/ecg_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Install library scipy jika belum ada (biasanya sudah bawaan Colab)
!pip install -q scipy ipywidgets

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import Audio, display, clear_output
import io
from scipy.io import wavfile

print("Library berhasil di-import!")
# 1. Fungsi dasar pembentuk komponen ECG
def gaussian_pulse(x, mu, sig, amp):
    return amp * np.exp(-(((x - mu) / sig) ** 2))

# 2. Fungsi utama simulator
def jalankan_simulasi_ecg(bpm=75, noise_level=0.02):
    clear_output(wait=True)

    # Parameter Sinyal
    fs = 500  # Frekuensi sampling (Hz)
    durasi = 5  # Durasi simulasi dalam detik
    t = np.linspace(0, durasi, fs * durasi)

    # Hitung kapan detak jantung harus muncul berdasarkan BPM
    detik_per_beat = 80.0 / bpm

    # Membuat sinyal ECG dasar (bersih)
    sinyal_bersih = np.zeros_like(t)
    puncak_r_teoretis = []

    # Loop untuk menyusun detak sepanjang durasi
    waktu_detak = 0.2
    while waktu_detak < durasi - 0.4:
        # Puncak R (Pusat detak)
        puncak_r_teoretis.append(waktu_detak)

        # Gabungan gelombang P, Q, R, S, T relatif terhadap puncak R
        sinyal_bersih += gaussian_pulse(t, waktu_detak - 0.12, 0.02, 0.15)  # P Wave
        sinyal_bersih += gaussian_pulse(t, waktu_detak - 0.03, 0.005, -0.1) # Q Wave
        sinyal_bersih += gaussian_pulse(t, waktu_detak, 0.01, 1.0)          # R Wave
        sinyal_bersih += gaussian_pulse(t, waktu_detak + 0.03, 0.008, -0.25)# S Wave
        sinyal_bersih += gaussian_pulse(t, waktu_detak + 0.18, 0.03, 0.35)  # T Wave

        waktu_detak += detik_per_beat

    # 3. Menambahkan Noise Gausian (Gangguan Sinyal)
    noise = np.random.normal(0, noise_level, size=len(t))
    sinyal_ecg = sinyal_bersih + noise

    # 4. Deteksi Peak Otomatis (Menggunakan Scipy)
    # Mencari puncak gelombang R yang lebih tinggi dari gelombang lainnya
    peaks, _ = find_peaks(sinyal_ecg, distance=fs*0.3, height=0.5)
    bpm_terdeteksi = int(len(peaks) * (60 / durasi))

    # 5. Membuat Audio Beep Medis Asli (1000 Hz)
    # Audio di-generate per detak yang terdeteksi
    fs_audio = 8000
    audio_total = np.zeros(int(fs_audio * durasi))

    for p in peaks:
        waktu_detik = t[p]
        indeks_audio_mulai = int(waktu_detik * fs_audio)
        durasi_beep = int(0.1 * fs_audio) # Beep selama 0.1 detik

        # Buat gelombang sinus 1000Hz untuk suara bip rumah sakit
        t_beep = np.linspace(0, 0.1, durasi_beep)
        beep = np.sin(2 * np.pi * 1000 * t_beep) * 0.5

        # Masukkan ke dalam timeline audio
        if indeks_audio_mulai + durasi_beep < len(audio_total):
            audio_total[indeks_audio_mulai : indeks_audio_mulai + durasi_beep] = beep

    # 6. Visualisasi Premium (Glow Effect & Grid Medis)
    plt.figure(figsize=(12, 5), facecolor='#0a0a0a')
    ax = plt.axes()
    ax.set_facecolor('#050d08') # Hijau tua gelap khas monitor

    # Membuat Grid Medis (Kotak-kotak ECG)
    ax.grid(True, which='both', color='#004422', linestyle='-', linewidth=0.5)
    ax.minorticks_on()
    ax.grid(True, which='minor', color='#002211', linestyle=':', linewidth=0.5)

    # Plot Sinyal ECG dengan efek agak menyala (Glow)
    plt.plot(t, sinyal_ecg, color='#00ffaa', linewidth=2, label='Sinyal ECG', zorder=3)

    # Plot Peak yang berhasil dideteksi (Tanda Merah/Orange di Puncak)
    plt.plot(t[peaks], sinyal_ecg[peaks], "x", color='#ff3344', markersize=10, markeredgewidth=3, label='Peak Terdeteksi', zorder=4)

    # Styling Konten Grafik
    status = "NORMAL" if bpm <= 100 else "TACHYCARDIA (Cepat)"
    if bpm < 50: status = "BRADYCARDIA (Lambat)"

    plt.title(f"MONITOR MEDIS VIRTUAL  |  Status: {status}", color='white', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Waktu (Detik)", color='#888888')
    plt.ylabel("Amplitudo (mV)", color='#888888')
    plt.tick_params(colors='white')
    plt.xlim(0, durasi)
    plt.ylim(-0.6, 1.5)

    # Tampilkan Legenda Medis
    plt.legend(loc='upper right', facecolor='#111111', edgecolor='#333333', labelcolor='white')

    # Tampilan Informasi Angka di Monitor
    plt.text(0.1, 1.2, f"SETTING BPM: {bpm}", color='#00ffaa', fontsize=12, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6))
    plt.text(1.2, 1.2, f"REAL-TIME DETECTED BPM: {bpm_terdeteksi}", color='#ffcc00', fontsize=12, fontweight='bold', bbox=dict(facecolor='black', alpha=0.6))

    plt.show()

    # 7. Mengeluarkan Output Suara
    display(Audio(audio_total, rate=fs_audio, autoplay=True))
    # Membuat widget slider interaktif murni menggunakan Python
interact(
    jalankan_simulasi_ecg,
    bpm=widgets.IntSlider(min=40, max=160, step=5, value=70, description='Heart Rate (BPM):'),
    noise_level=widgets.FloatSlider(min=0.0, max=0.6, step=0.05, value=0.05, description='Noise Level:')
);


Library berhasil di-import!


interactive(children=(IntSlider(value=70, description='Heart Rate (BPM):', max=160, min=40, step=5), FloatSlid…

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Audio, display, clear_output

def gaussian_pulse(x, mu, sig, amp):
    return amp * np.exp(-(((x - mu) / sig) ** 2))

def jalankan_simulasi_ecg_v3(bpm=80, noise_level=0.02):
    clear_output(wait=True)

    fs = 250
    durasi = 5
    t = np.linspace(0, durasi, fs * durasi)
    detik_per_beat = 60.0 / bpm

    sinyal_bersih = np.zeros_like(t)

    waktu_detak = 0.3
    while waktu_detak < durasi - 0.2:
        sinyal_bersih += gaussian_pulse(t, waktu_detak - 0.12, 0.02, 0.15)  # P
        sinyal_bersih += gaussian_pulse(t, waktu_detak - 0.03, 0.005, -0.1) # Q
        sinyal_bersih += gaussian_pulse(t, waktu_detak, 0.012, 1.0)         # R
        sinyal_bersih += gaussian_pulse(t, waktu_detak + 0.03, 0.008, -0.25)# S
        sinyal_bersih += gaussian_pulse(t, waktu_detak + 0.18, 0.03, 0.35)  # T
        waktu_detak += detik_per_beat

    noise = np.random.normal(0, noise_level, size=len(t))
    sinyal_ecg = sinyal_bersih + noise

    # Deteksi Peak Adaptif
    tinggi_maksimum = np.max(sinyal_ecg)
    batas_adaptif = tinggi_maksimum * 0.6 if tinggi_maksimum > 0.5 else 0.4
    peaks, _ = find_peaks(sinyal_ecg, distance=int(fs * 0.25), height=batas_adaptif)

    # --- BERIKUT VARIABEL TOTAL PEAK & PERHITUNGAN BPM ---
    total_peak_terdeteksi = len(peaks) # Menghitung total peak asli di grafik
    bpm_terdeteksi = int(total_peak_terdeteksi * (60 / durasi))
    # -----------------------------------------------------

    # Generate Audio Suara Beep
    fs_audio = 8000
    audio_total = np.zeros(int(fs_audio * durasi))
    for p in peaks:
        waktu_detik = t[p]
        indeks_audio_mulai = int(waktu_detik * fs_audio)
        durasi_beep = int(0.08 * fs_audio)
        t_beep = np.linspace(0, 0.08, durasi_beep)
        beep = np.sin(2 * np.pi * 1000 * t_beep) * 0.4
        if indeks_audio_mulai + durasi_beep < len(audio_total):
            audio_total[indeks_audio_mulai : indeks_audio_mulai + durasi_beep] = beep

    # Visualisasi Monitor Medis Premium
    plt.figure(figsize=(12, 5), facecolor='#0a0a0a')
    ax = plt.axes()
    ax.set_facecolor('#050d08')

    ax.grid(True, which='both', color='#004422', linestyle='-', linewidth=0.5)
    ax.minorticks_on()
    ax.grid(True, which='minor', color='#002211', linestyle=':', linewidth=0.5)

    # Plot Sinyal & Peak
    plt.plot(t, sinyal_ecg, color='#00ffaa', linewidth=2.5, label='Sinyal ECG', zorder=3)
    if total_peak_terdeteksi > 0:
        plt.plot(t[peaks], sinyal_ecg[peaks], "x", color='#ff3344', markersize=12, markeredgewidth=3, label='Peak Terdeteksi (R-Wave)', zorder=4)

    status = "NORMAL" if bpm <= 100 else "TACHYCARDIA"
    if bpm < 50: status = "BRADYCARDIA"

    plt.title(f"MONITOR MEDIS VIRTUAL  |  Status: {status}", color='white', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Waktu (Detik)", color='#888888')
    plt.ylabel("Amplitudo (mV)", color='#888888')
    plt.tick_params(colors='white')
    plt.xlim(0, durasi)
    plt.ylim(-0.8, 1.8)

    plt.legend(loc='upper right', facecolor='#111111', edgecolor='#333333', labelcolor='white')

    # --- MENAMPILKAN TOTAL PEAK DAN BPM DI MONITOR ---
    plt.text(0.1, 1.5, f"SETTING BPM: {bpm}", color='#00ffaa', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.7))
    plt.text(1.1, 1.5, f"TOTAL PEAK: {total_peak_terdeteksi}", color='#00ffb7', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.7))
    plt.text(2.1, 1.5, f"REAL-TIME BPM: {bpm_terdeteksi}", color='#ffcc00', fontsize=11, fontweight='bold', bbox=dict(facecolor='black', alpha=0.7))
    # -------------------------------------------------

    plt.show()
    display(Audio(audio_total, rate=fs_audio, autoplay=True))

# Jalankan Interaktif
interact(
    jalankan_simulasi_ecg_v3,
    bpm=widgets.IntSlider(min=45, max=150, step=5, value=75, description='Heart Rate (BPM):'),
    noise_level=widgets.FloatSlider(min=0.0, max=0.5, step=0.05, value=0.05, description='Noise Level:')
);

interactive(children=(IntSlider(value=75, description='Heart Rate (BPM):', max=150, min=45, step=5), FloatSlid…